# NB162 — Inter-Generation Derivation

**Discovery (NB161)**: x(R0) = Product(1-f_k) × P3 = phi(p3)/p4 = 4/7.
The mass exponent is the coherent fraction across all 4 levels × generation spacing.

**Goal**: Apply the same formula to the 2->3 generation step. The 2->3 step
involves different crossing positions with different wrapping fractions.
If the formula generalizes, we can derive r_bs and r_tc from first principles.

**Key question**: The 1->2 step uses the CP pair (ci=11 vs ci=191).
What crossing pair does the 2->3 step use, and what are its wrapping fractions?

In [2]:
# S0 — Wrapping fractions at ALL quark generation crossings
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = 1.0 / np.sqrt(P4)
P3 = p1 * p2 * p3  # 30

cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)

from solenoid_jax import warmup
warmup()

sys0 = SolenoidSystem(primes=primes)
branches = sys0.all_branches()
res = sys0.integrate_all_branches(branches, cis.astype(float), float(P4+1), backend='jax')

# Compute wrapping fractions at ALL a5=0 quark crossings, ALL levels
gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}
n_br = len(branches)

print('=== Wrapping fractions at all quark (a3=1, a5=0) crossings ===')
print(f'{"ci":>4} {"a7":>3} {"gen":>4} {"Z2":>3}  {"f_R0":>8} {"f_R1":>8} {"f_R2":>8} {"f_R3":>8}  {"Prod(1-f)":>10} {"×P3":>8}')
print('-' * 85)

crossing_data = {}
for idx in range(len(cis)):
    if a5[idx] != 0 or a3[idx] != 1:
        continue
    ci_val = cis[idx]
    gen = gen_map[a7[idx]]
    z2 = a7[idx] % 2
    
    fracs = []
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in branches])
        f = np.sum(np.abs(Rk) > np.pi) / n_br
        fracs.append(f)
    
    nwf = [(1-f) for f in fracs]  # non-wrapping fractions
    prod_nwf = np.prod(nwf)
    x_pred = prod_nwf * P3
    
    crossing_data[(gen, z2)] = {
        'ci': ci_val, 'fracs': fracs, 'nwf': nwf,
        'prod_nwf': prod_nwf, 'x_pred': x_pred
    }
    
    print(f'{ci_val:4d} {a7[idx]:3d} {gen:4d} {z2:3d}  '
          f'{fracs[0]*100:7.1f}% {fracs[1]*100:7.1f}% {fracs[2]*100:7.1f}% {fracs[3]*100:7.1f}%  '
          f'{prod_nwf:10.6f} {x_pred:8.4f}')

# ══════════════════════════════════════════════════════════════
# NB161 formula: x(R0) = Product(1-f_k) × P3
# Applied to gen2 Z2=0 (ci=11): gives 4/7 = 0.5714 ✓
# What does it give at OTHER crossings?
# ══════════════════════════════════════════════════════════════

print(f'\n=== NB161 formula at each crossing ===')
print(f'Formula: x_pred = Product(1-f_k) × P3')
print(f'\nZ2=0 quarks (the CP pair coset):')
for gen in [1, 2, 3]:
    d = crossing_data[(gen, 0)]
    print(f'  gen{gen} (ci={d["ci"]}): x_pred = {d["x_pred"]:.6f}  '
          f'nwf = [{d["nwf"][0]:.4f}, {d["nwf"][1]:.4f}, {d["nwf"][2]:.4f}, {d["nwf"][3]:.4f}]')

print(f'\nZ2=1 quarks:')
for gen in [1, 2, 3]:
    d = crossing_data[(gen, 1)]
    print(f'  gen{gen} (ci={d["ci"]}): x_pred = {d["x_pred"]:.6f}  '
          f'nwf = [{d["nwf"][0]:.4f}, {d["nwf"][1]:.4f}, {d["nwf"][2]:.4f}, {d["nwf"][3]:.4f}]')

# ══════════════════════════════════════════════════════════════
# The inter-gen mass ratio might come from the RATIO of x_pred
# at different generation crossings.
# ══════════════════════════════════════════════════════════════

print(f'\n=== Inter-gen ratios from the formula ===')

# The 1->2 step: x_q = x_pred at gen2 = 0.5714 = 4/7
# The mass ratio m_s/m_d = CP_R0^{x_q} = 20.0
#
# For the 2->3 step: what exponent gives m_b/m_s?
# Hypothesis 1: x_{bs} = x_pred at gen3 crossing?
# Hypothesis 2: x_{bs} = x_pred(gen3) - x_pred(gen2)?
# Hypothesis 3: x_{bs} = x_pred(gen2) + x_pred(gen1)?

x_g1_z0 = crossing_data[(1, 0)]['x_pred']
x_g2_z0 = crossing_data[(2, 0)]['x_pred']
x_g3_z0 = crossing_data[(3, 0)]['x_pred']

x_g1_z1 = crossing_data[(1, 1)]['x_pred']
x_g2_z1 = crossing_data[(2, 1)]['x_pred']
x_g3_z1 = crossing_data[(3, 1)]['x_pred']

print(f'x_pred values (Z2=0): gen1={x_g1_z0:.6f}, gen2={x_g2_z0:.6f}, gen3={x_g3_z0:.6f}')
print(f'x_pred values (Z2=1): gen1={x_g1_z1:.6f}, gen2={x_g2_z1:.6f}, gen3={x_g3_z1:.6f}')

# The scaling factors:
# r_bs = x_{bs}(R0) / x_q(R0) = x_{bs} / (4/7)
# We need r_bs = 1.269, so x_{bs}(R0) = 1.269 × 4/7 = 0.7251
# And r_tc = 1.639, so x_{tc}(R0) = 1.639 × 4/7 = 0.9368

x_bs_target = 1.2688 * 4/7  # 0.7251
x_tc_target = 1.6394 * 4/7  # 0.9368
print(f'\nTarget R0 exponents:')
print(f'  1->2 (m_s/m_d): {4/7:.6f} = 4/7 [known]')
print(f'  2->3 down (m_b/m_s): {x_bs_target:.6f}')
print(f'  2->3 up (m_t/m_c): {x_tc_target:.6f}')

# Test all possible combinations of x_pred values:
print(f'\n--- Testing combinations of x_pred for m_b/m_s target {x_bs_target:.4f} ---')
vals = {
    'g1z0': x_g1_z0, 'g2z0': x_g2_z0, 'g3z0': x_g3_z0,
    'g1z1': x_g1_z1, 'g2z1': x_g2_z1, 'g3z1': x_g3_z1,
}

# Sums
for na, va in vals.items():
    for nb, vb in vals.items():
        if na >= nb:
            continue
        s = va + vb
        dev = abs(s - x_bs_target) / x_bs_target * 100
        if dev < 5:
            print(f'  {na} + {nb} = {va:.4f} + {vb:.4f} = {s:.4f} ({dev:.2f}% from target)')

# Products
for na, va in vals.items():
    for nb, vb in vals.items():
        if na >= nb:
            continue
        p = va * vb
        dev = abs(p - x_bs_target) / x_bs_target * 100
        if dev < 5:
            print(f'  {na} × {nb} = {va:.4f} × {vb:.4f} = {p:.4f} ({dev:.2f}% from target)')

# Simple multiples
for na, va in vals.items():
    for m in [0.5, 1, 1.5, 2, 3]:
        v = va * m
        dev = abs(v - x_bs_target) / x_bs_target * 100
        if dev < 3:
            print(f'  {m}×{na} = {m}×{va:.4f} = {v:.4f} ({dev:.2f}% from target)')

# Ratios
for na, va in vals.items():
    for nb, vb in vals.items():
        if na == nb or vb == 0:
            continue
        r = va / vb
        dev = abs(r - x_bs_target) / x_bs_target * 100
        if dev < 5:
            print(f'  {na}/{nb} = {va:.4f}/{vb:.4f} = {r:.4f} ({dev:.2f}% from target)')

print(f'\n--- Testing combinations for m_t/m_c target {x_tc_target:.4f} ---')
for na, va in vals.items():
    for nb, vb in vals.items():
        if na >= nb:
            continue
        s = va + vb
        dev = abs(s - x_tc_target) / x_tc_target * 100
        if dev < 5:
            print(f'  {na} + {nb} = {va:.4f} + {vb:.4f} = {s:.4f} ({dev:.2f}% from target)')

for na, va in vals.items():
    for nb, vb in vals.items():
        if na >= nb:
            continue
        p = va * vb
        dev = abs(p - x_tc_target) / x_tc_target * 100
        if dev < 5:
            print(f'  {na} × {nb} = {va:.4f} × {vb:.4f} = {p:.4f} ({dev:.2f}% from target)')

for na, va in vals.items():
    for m in [0.5, 1, 1.5, 2, 3]:
        v = va * m
        dev = abs(v - x_tc_target) / x_tc_target * 100
        if dev < 3:
            print(f'  {m}×{na} = {m}×{va:.4f} = {v:.4f} ({dev:.2f}% from target)')

for na, va in vals.items():
    for nb, vb in vals.items():
        if na == nb or vb == 0:
            continue
        r = va / vb
        dev = abs(r - x_tc_target) / x_tc_target * 100
        if dev < 5:
            print(f'  {na}/{nb} = {va:.4f}/{vb:.4f} = {r:.4f} ({dev:.2f}% from target)')

# ══════════════════════════════════════════════════════════════
# Also try: the inter-gen exponent involves BOTH Z2 cosets.
# The down-type (m_b/m_s) involves the down-type isospin.
# The up-type (m_t/m_c) involves the up-type isospin.
# Maybe the Z2 cosets correspond to up/down isospin.
# ══════════════════════════════════════════════════════════════

print(f'\n=== Z2 coset sums and cross-coset combinations ===')
# Sum of Z2=0 and Z2=1 at each generation:
for gen in [1, 2, 3]:
    x_z0 = crossing_data[(gen, 0)]['x_pred']
    x_z1 = crossing_data[(gen, 1)]['x_pred']
    print(f'  gen{gen}: Z2=0={x_z0:.6f}, Z2=1={x_z1:.6f}, '
          f'sum={x_z0+x_z1:.6f}, mean={0.5*(x_z0+x_z1):.6f}, '
          f'product={x_z0*x_z1:.6f}')

# Cross-generation cross-coset:
print(f'\nCross-gen cross-coset combinations:')
print(f'  g2z0 + g3z1 = {x_g2_z0 + x_g3_z1:.6f}  (target m_b/m_s: {x_bs_target:.6f})')
print(f'  g2z1 + g3z0 = {x_g2_z1 + x_g3_z0:.6f}')
print(f'  g1z0 + g2z1 = {x_g1_z0 + x_g2_z1:.6f}')
print(f'  g1z1 + g2z0 = {x_g1_z1 + x_g2_z0:.6f}  (target m_t/m_c: {x_tc_target:.6f})')
print(f'  g1z1 + g3z0 = {x_g1_z1 + x_g3_z0:.6f}')
print(f'  g1z0 + g3z1 = {x_g1_z0 + x_g3_z1:.6f}')

  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.56s
=== Wrapping fractions at all quark (a3=1, a5=0) crossings ===
  ci  a7  gen  Z2      f_R0     f_R1     f_R2     f_R3   Prod(1-f)      ×P3
-------------------------------------------------------------------------------------
  11   4    2   0      0.0%    50.0%    73.3%    85.7%    0.019048   0.5714
  41   3    1   1      0.0%     0.0%     0.0%     7.1%    0.928571  27.8571
  71   0    1   0      0.0%     0.0%     0.0%     0.0%    1.000000  30.0000
 101   1    2   1      0.0%     0.0%     0.0%     0.0%    1.000000  30.0000
 131   5    3   1      0.0%     0.0%     0.0%     0.0%    1.000000  30.0000
 191   2    3   0      0.0%     0.0%     0.0%     0.0%    1.000000  30.0000

=== NB161 formula at each crossing ===
Formula: x_pred = Product(1-f_k) × P3

Z2=0 quarks (the CP pair coset):
  gen1 (ci=71): x_pred = 30.000000  nwf = [1.0000, 1.0000, 1.0000, 1.0000]
  gen2 (ci=11): x_pred = 0.571429  nwf = [1.0000, 0.5000, 0.26